# Stage 2: core C3 (T̂ vs S_cat vs aggregator vs blind baseline), Enron, GPT-Neo 125M
**Plan:** section 4 row 2, Steps D–F. 10 main seeds (4 epochs), plus a null model for seed 0.

**Pre-registered headline (fixed 2026-11-24, before any Stage 2 result):** T̂, m = 32, η = 0.05, all 175 users, main run, **seed 1**,
DKW certificate (Prop. 2 ii). Everything else in this notebook is descriptive.

**Getting the results:** download each Kaggle archive `/kaggle/working/outputs/S2_<time>.tar.gz` and extract it into `Stage2/results/`.
Model weights are not needed here.

In [ ]:
import os, sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath(".."))
import analysis as A
RESULTS = os.environ.get("S2_RESULTS", "results")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
# categorical slots 1-4 (validated: CVD and normal-vision separation pass; contrast WARN -> tables shown next to every plot)
COL = {"T": "#2a78d6", "S_cat": "#eb6834", "Agg": "#1baf7a", "Blind": "#eda100"}
MRK = {"T": "o", "S_cat": "s", "Agg": "^", "Blind": "D"}
NAME = {"T": "T̂ (mean)", "S_cat": "S_cat", "Agg": "aggregator", "Blind": "blind (text only)"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25})

sc, cat, hist = A.load(RESULTS)
texts, users = A.load_texts()
runs = sc.groupby(["run", "seed", "user"]).agg(b=("trained", "first"), best_epoch=("best_epoch", "first")).reset_index()
display(runs.groupby(["run", "seed"]).agg(users=("user", "size"), trained=("b", "sum"), best_epoch=("best_epoch", "first")))
print("S_cat tables:", sorted(set(map(tuple, cat[["run", "seed"]].drop_duplicates().to_numpy()))) if cat is not None else None)

## 1. Training
Best epoch (lowest member validation loss) per run, out of 4.

In [ ]:
H = A.history_table(hist)
if H.empty:
    print("no history files")
else:
    display(H[H.best].round(4).reset_index(drop=True))
    display(H.pivot_table(index="run", columns="epoch", values="val_in").round(3))

## 2. Headline certificate (pre-registered)

In [ ]:
h = A.headline(sc)
print("seed 1 main run not available yet" if h is None else
      {k: (np.round(v, 3).tolist() if isinstance(v, (float, list)) else v) for k, v in h.items()})

## 3. All statistics vs m
Per (run, seed, m, draw): the same m-document subsets for every statistic (`common/draws.py`). Metrics are averaged over the draws (4 for m < 32, 1 at m = 32), then over seeds.
- `users = all`: T̂ and S_cat on all 175 users.
- `users = EVAL`: all four statistics on the 88 EVAL users. The aggregator and the blind baseline are fit on FIT.

`dkw_cert` is J-hat minus the DKW terms. `fixed_cert` is Prop. 2(i), with the threshold chosen on FIT. Running the evaluation takes a few minutes.

In [ ]:
ev = A.evaluate(sc, cat, texts=texts, eta=0.05)
per_seed, agg = A.summarize(ev)
ev.to_csv(os.path.join(RESULTS, "_eval_long.csv.gz"), index=False)
display(agg[agg.run == "main"].round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, g in zip(axes, ["all", "EVAL"]):
    for st in A.STATS:
        x = per_seed[(per_seed.run == "main") & (per_seed.users == g) & (per_seed.stat == st)]
        if x.empty: continue
        s = x.groupby("m").auroc.agg(["mean", "min", "max"])
        ax.fill_between(s.index, s["min"], s["max"], color=COL[st], alpha=.15, lw=0)
        ax.plot(s.index, s["mean"], marker=MRK[st], color=COL[st], lw=2, ms=6, label=NAME[st])
    ax.axhline(.5, color="#888", lw=1, ls=":")
    ax.set_xscale("log", base=2); ax.set_xticks(A.DR.M_GRID, A.DR.M_GRID)
    ax.set_xlabel("m (held-out docs per user)"); ax.set_title(f"main run, {g} users (mean over seeds; band = min–max)", fontsize=10)
axes[0].set_ylabel("AUROC")
hs, ls = axes[1].get_legend_handles_labels()
fig.legend(hs, ls, frameon=False, loc="lower center", ncol=4)
plt.tight_layout(rect=(0, 0.08, 1, 1)); plt.show()

## 4. Does any statistic beat T̂? (falsifier 2)
The paired difference AUROC(stat) − AUROC(T̂) on EVAL users, per seed. The spread over seeds **understates** the uncertainty over users, because every seed reuses the same 175 users.

In [ ]:
display(A.paired_vs_T(per_seed, "EVAL").round(3))
display(A.paired_vs_T(per_seed[per_seed.stat.isin(["T", "S_cat"])], "all").round(3))

## 5. Variance decomposition, Prop. 6 (falsifiers 1 and 3)
Per seed: Δ, v_B and v_W for each class. Predicted AUROC(m) = Φ(Δ / √(v_B¹ + v_B⁰ + (v_W¹ + v_W⁰)/m)), and the ceiling at m → ∞.
These are compared with the measured T̂ AUROC (200 random subsets per m). Eq. (3) is an approximation (Gaussian), so a mismatch is informative, not necessarily a failure.

In [ ]:
seeds = sorted(sc[sc.run == "main"].seed.unique())
vd = [A.variance_decomposition(sc, "main", s) for s in seeds]
meas = {s: A.measured_T_auroc(sc, "main", s, R=200) for s in seeds}
rows = [{"seed": v["seed"], "m": m, "predicted": v["pred"][m], "measured": meas[v["seed"]][m], "ceiling": v["ceiling"]}
        for v in vd for m in A.DR.M_GRID]
P = pd.DataFrame(rows)
display(pd.DataFrame([{k: v[k] for k in ("seed", "delta", "v_B1", "v_B0", "v_W1", "v_W0", "ceiling", "share_between_at_m32")} for v in vd]).round(3))
display(P.groupby("m")[["predicted", "measured", "ceiling"]].mean().round(3))
fig, ax = plt.subplots(figsize=(5.5, 3.6))
s = P.groupby("m").mean()
ax.plot(s.index, s.measured, marker="o", color=COL["T"], lw=2, label="measured T̂")
ax.plot(s.index, s.predicted, marker="s", color=COL["S_cat"], lw=2, ls="--", label="predicted, eq. (3)")
ax.axhline(s.ceiling.iloc[0], color="#888", lw=1, ls=":", label="predicted ceiling")
ax.set_xscale("log", base=2); ax.set_xticks(A.DR.M_GRID, A.DR.M_GRID); ax.set_xlabel("m"); ax.set_ylabel("AUROC")
ax.set_title("Prop. 6: predicted vs measured (mean over seeds)", fontsize=10); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 6. Null model (seed 0, EVAL users never trained on, fake labels)
Every statistic must show J ≈ 0: the certificate is ≤ 0 at every m. The aggregator is fit on FIT users, who carry real training signal in the null model, and applied to EVAL.

In [ ]:
nl = per_seed[per_seed.run == "null"]
if nl.empty:
    print("no null run yet")
else:
    display(nl.pivot_table(index="m", columns="stat", values=["auroc", "dkw_cert"]).round(3))
    bad = ev[(ev.run == "null") & (ev.dkw_cert > 0)]
    print("null PASS: no positive DKW certificate in any (stat, m, draw)" if bad.empty else "null FAIL:")
    if not bad.empty: display(bad)

## 7. Blind baseline (text only, no model)
Under a random user-level split, it must stay at chance. A positive certificate would mean the split or the preprocessing leaks.

In [ ]:
b = per_seed[(per_seed.run == "main") & (per_seed.stat == "Blind")]
display(b.groupby("m")[["auroc", "j_hat", "dkw_cert"]].agg(["mean", "max"]).round(3))

## Notes (to fill after reading the results)
- Training (best epochs):
- Headline certificate (seed 1):
- T̂ vs S_cat vs aggregator (falsifier 2):
- Saturation and the predicted ceiling (falsifier 1):
- Predicted vs measured AUROC (falsifier 3):
- Null model:
- Blind baseline:
- Decision: